# Poisson problem on a Pacman domain

We solve

$$
\begin{align*}
-\Delta u=f\quad\text{in }\Omega_\theta,\qquad u=0\quad\text{on } \partial\Omega_\theta,
\end{align*}
$$

on the circular sector

$$
\begin{align*}
\Omega_\theta=\{r(\cos\varphi,\sin\varphi):0<r<1,\ 0<\varphi<\theta\}, \qquad 0<\theta<2\pi.
\end{align*}
$$

The parameter $\theta$ is the **interior angle**, measured in radians. Let

$$
\begin{align*}
\beta=\frac{\pi}{\theta},\qquad u(r,\varphi)=(1-r^2)r^\beta\sin(\beta\varphi),\qquad f(r,\varphi)=4(\beta+1)r^\beta\sin(\beta\varphi).
\end{align*}
$$

## Packages

For the error plot, **Matplotlib** is needed in addition to NGSolve/Netgen (`pip install matplotlib`).

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import WorkPlane, OCCGeometry

## Geometry and finite element space

All boundary edges are called `outer`, so `dirichlet="outer"` imposes the homogeneous Dirichlet condition on the entire boundary. Use `mesh.Curve(order)` to curve the boundary to avoid a geometry discretization error to dominate.

In [ ]:
def pacman_geometry(theta):
    """Sector of radius 1 with interior angle theta (in radians)"""
    if not 0 < theta < 2*pi:
        raise ValueError("theta must lie between 0 and 2*pi.")
    face = (WorkPlane().MoveTo(0, 0).LineTo(1, 0).Rotate(90)
            .Arc(1, theta/pi*180).Close().Face())
    face.edges.name = "outer"
    face.faces.name = "sector"
    return OCCGeometry(face, dim=2)

In [ ]:
theta = 3*pi/2
order = 1
maxh = 0.3

geo = pacman_geometry(theta)
mesh = Mesh(geo.GenerateMesh(maxh=maxh))
mesh.Curve(order)
Draw(mesh);

## 1. Exact solution and source

Use the angular coordinate in $[0,2\pi)$, including the lower half-plane.

In [ ]:
def exact_data(theta):
    beta = pi/theta
    r = sqrt(x*x+y*y)
    # atan2 alone returns negative angles in the lower half-plane
    phi = IfPos(-y, atan2(y, x)+2*pi, atan2(y, x))
    s, c = sin(beta*phi), cos(beta*phi)
    exact = (1-r*r)*r**beta*s
    rhs = 4*(beta+1)*r**beta*s
    # grad(u) = u_r*e_r + (u_phi/r)*e_phi
    radial = (beta-(beta+2)*r*r)*r**(beta-1)*s
    angular = beta*(1-r*r)*r**(beta-1)*c
    exact_grad = CF((radial*cos(phi)-angular*sin(phi),
                     radial*sin(phi)+angular*cos(phi)))
    return exact, exact_grad, rhs

exact, exact_grad, rhs = exact_data(theta)

Draw(exact, mesh)
# Optional gradient visualization: the guard is only for drawing near r=0
Draw(Norm(IfPos(sqrt(x*x+y*y)-1e-8,exact_grad,CF((2,2)))), mesh, order=3, deformation=True)
Draw(rhs, mesh);

## 2. Finite element solution

Write the weak form and assemble its bilinear and linear forms using the provided `fes` and its trial and test functions `u, v`. Solve for a `GridFunction` called `uh`, respecting `fes.FreeDofs()`, and display the numerical solution.

Use the assembly and solve cell in the [October 14 Laplace notebook](../../notebooks/VO_2026_10_14/01_simple_laplace.ipynb) as a reference.

In [ ]:
fes = H1(mesh, order=order, dirichlet="outer")
u, v = fes.TnT()

# TODO: Assemble and solve the discrete Poisson problem.
# TODO: Display uh with Draw(uh, mesh).

## 3. Error norms

Compute both errors

$$
\begin{align*}
\|u-u_h\|_{L^2}\qquad \text{and}\qquad \|u-u_h\|_{H^1}=\sqrt{\|u-u_h\|_{L^2}^2+\|\nabla u-\nabla u_h\|_{L^2}^2}.
\end{align*}
$$

For a vector-valued coefficient function `g`, the product `g*g` is its pointwise scalar product. For example,

```python
g = exact_grad - grad(uh)
gradient_error_sq = Integrate(g*g, mesh)
```

`Norm(g)` alone gives a pointwise magnitude. A spatial norm also requires integration.

In [ ]:
# TODO: Compute L2_error and H1_error.

## 4. Convergence experiments

Repeat the computation for $\theta\in\{\pi/2,3\pi/2,7\pi/4\}$ and `order` in $\{1, 2, 3\}$. Generate a new mesh and a new finite element space for each smaller `maxh`, for example `[0.4, 0.2, 0.1, 0.05, 0.025]`. Keep the geometry degree consistent by calling `mesh.Curve(order)`. Here $h$ denotes the chosen `maxh`, as in the exercise.

Collect `hs`, `L2_errors`, and `H1_errors` for each angle and order. Estimate rates from consecutive entries using

$$
\begin{align*}
s=\frac{\log(e_i/e_{i+1})}{\log(h_i/h_{i+1})}.
\end{align*}
$$

Explain the difference between the convex and reentrant sectors.

For each fixed angle and polynomial degree, start with empty error lists. For each `maxh`:

1. Create a new mesh from the geometry and call `mesh.Curve(order)`
2. Create a new `fes` and its trial and test functions
3. Assemble the forms on this space and solve for a new `uh`
4. Compute both errors on this mesh and append the results

```python
hs.append(maxh)
L2_errors.append(L2_error)
H1_errors.append(H1_error)
```

Entries with the same index must come from the same computation. Changing `maxh` or `order` alone does not update an existing mesh or finite element space. When changing `theta`, also rebuild the geometry and call `exact_data(theta)` again. You may put repeated solve and error calculations into functions.

For example, print the rates between consecutive meshes using

```python
for i in range(len(hs)-1):
    rate = log(H1_errors[i]/H1_errors[i+1]) / log(hs[i]/hs[i+1])
    print(f"h={hs[i]:g} -> {hs[i+1]:g}: rate={rate:.3f}")
```

In [ ]:
# TODO: Fill these lists for one fixed angle and polynomial degree.
hs = []
L2_errors = []
H1_errors = []

# TODO: Repeat for all angles and polynomial degrees in the exercise.

## Simple Matplotlib example for the $H^1$ error

After filling `hs` and `H1_errors`, copy the following into a code cell:

```python
import matplotlib.pyplot as plt

plt.figure()
plt.loglog(hs, H1_errors, "o-", label=f"order={order}")
plt.xlabel("h = maxh")
plt.ylabel(r"$\|u-u_h\|_{H^1}$")
plt.grid(True, which="both")
plt.legend()
plt.show()
```

`plt.figure()` starts a new figure. Each `plt.loglog(...)` adds a curve with logarithmic axes, and `"o-"` draws markers connected by lines. The `label` appears in the legend.

To compare several polynomial degrees for one angle, call `plt.figure()` once before the loop over degrees. After filling the lists for each degree, add its curve with `plt.loglog(...)`. Put the axis labels, grid, `plt.legend()`, and `plt.show()` after that loop, so all curves appear together. Start a new figure for another angle or error norm. Use the same pattern with `L2_errors` for the $L^2$ plot.

A reference curve $C h^s$ has slope $s$ in a log-log plot. The positive constant $C$ only changes its vertical position. For example, to compare with linear convergence, insert this before `plt.legend()` and `plt.show()`:

```python
s = 1
C = H1_errors[0]/hs[0]**s
reference_errors = [C*h**s for h in hs]
plt.loglog(hs, reference_errors, "k--", label=f"reference slope {s:g}")
```

Here the reference curve passes through the first error value. `"k--"` gives a black dashed line. Change `s` to compare with other candidate rates. Compare the curves and report the observed convergence rates.